# 04 - Robustness sweep: more findings, prompt wordings, calibration-free estimand

**Why this notebook.** Notebook 03 found, for Effusion: MedGemma R_dir 0.150 > CheXpert DenseNet 0.063 > Qwen-7B ~0.
Three things could still make that result fragile:
1. **One finding only.** Does the ordering hold for Atelectasis, Pneumothorax, Cardiomegaly?
   (Pneumothorax is interesting on purpose: models are known to use chest drains as a shortcut,
   and a drain often stays in after the pneumothorax resolves.)
2. **One prompt only.** A VLM's null could be a prompt failure; its positive could be a wording artefact.
   Effusion is re-run with two more wordings, one of them **negated** ("Is this X-ray free of...?"),
   which reverses the meaning of "yes" and exposes a yes/no answer bias.
3. **MedGemma's saturated scores.** R_sign counts only the *direction* of movement, so it is unchanged by any
   monotone recalibration. Items where yes/no gets < 90% of the next-token mass are also dropped in a sensitivity analysis.

**What runs** (all with the exact-gap pair controls from notebook 03):
- DenseNet: all findings, up to 1,500 balanced anchors each, plus a 1,000-image random AUC check per finding.
- Qwen2.5-VL-7B and MedGemma-4B: prompt P0 on 400 balanced anchors per finding; prompts P1, P2 on Effusion only.

**Settings:** NIH Chest X-rays input (Datasets tab), **GPU T4 x2**, Internet on, secret **HF_TOKEN**.
**Optional but recommended:** also attach the *output of notebook 03* (Add Input -> Notebooks -> your notebook 03).
Its Effusion scores are re-used, which saves about 1 GPU hour.

**Runtime:** about 6-9 GPU hours. Use **Save Version -> Save & Run All (commit)** so it runs in the background (12 h limit).
A time budget (`TIME_BUDGET_H`) stops scoring early and still writes results. If it stops early, attach this
notebook's own output as an input and run it again: cached scores are picked up and only the rest is scored.

In [1]:
!pip -q install torchxrayvision
# If a VLM fails with an unknown-architecture error:  !pip -q install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 64.6 MB/s eta 0:00:00


In [2]:
%%writefile pilot_lib.py
# ---- Pair construction and estimation (no model code) ----
import numpy as np
import pandas as pd

NIH_FINDINGS = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Effusion", "Emphysema",
                "Fibrosis", "Hernia", "Infiltration", "Mass", "Nodule", "Pleural_Thickening",
                "Pneumonia", "Pneumothorax"]


def add_label(df, finding):
    out = df.copy()
    out["label"] = out["Finding Labels"].str.split("|").apply(lambda L: int(finding in L))
    return out


def build_anchor_triples(df, finding, same_view=True, max_anchors_per_patient=2, seed=0,
                         match_gap=False, gap_tol=1, control="anchor"):
    """For each anchor image A, find in the SAME patient:
       flip partner  F: another study with the opposite label for `finding` (should change the answer)
       same partner  S: another study with the same label (should not change the answer)
    Partners are the nearest follow-ups (smallest |follow-up gap|) and share A's view if same_view.
    Using a same-patient, different-day image as the control means both arms carry the same kind
    of nuisance change (positioning, exposure, time), so R = S - N subtracts it."""
    rng = np.random.default_rng(seed)
    d = add_label(df, finding)
    key = ["Patient ID", "View Position"] if same_view else ["Patient ID"]
    d = d[d.groupby(key)["Image Index"].transform("size") >= 3]
    rows = []
    for k, g in d.groupby(key):
        pid = k[0] if isinstance(k, tuple) else k
        img = g["Image Index"].to_numpy(); lab = g["label"].to_numpy()
        fu = g["Follow-up #"].to_numpy(); view = g["View Position"].to_numpy()
        if lab.min() == lab.max():
            continue
        anchors = []
        for i in range(len(g)):
            other = np.arange(len(g)) != i
            fl = np.where(other & (lab != lab[i]))[0]
            sm = np.where(other & (lab == lab[i]))[0]
            if len(fl) == 0 or len(sm) == 0:
                continue
            f = fl[np.abs(fu[fl] - fu[i]).argmin()]
            if control == "pair":
                # control = a same-label PAIR (u, v) of this patient with EXACTLY the flip pair's signed
                # follow-up gap; prefer the anchor's status, then the pair closest in time to the anchor
                k = fu[f] - fu[i]
                best = None
                for u in range(len(g)):
                    for v in np.where((fu - fu[u]) == k)[0]:
                        if v == u or lab[u] != lab[v] or {u, v} == {i, f}:
                            continue
                        key_ = (lab[u] != lab[i], abs(fu[u] - fu[i]))
                        if best is None or key_ < best[0]:
                            best = (key_, u, v)
                if best is None:
                    continue
                _, u, v = best
                anchors.append(dict(patient=pid, view=view[i], img_a=img[i], lab_a=int(lab[i]),
                                    img_f=img[f], lab_f=int(lab[f]), img_s1=img[u], img_s=img[v],
                                    lab_s=int(lab[v]), gap_f=int(abs(k)), gap_s=int(abs(fu[v] - fu[u])),
                                    sgap_f=int(k), sgap_s=int(fu[v] - fu[u])))
                continue
            if match_gap:
                # control at the same signed time offset as the flip partner (same direction in time,
                # similar distance), so time/positioning drift is equal in both arms
                off = (fu[sm] - fu[i]) - (fu[f] - fu[i])
                s = sm[np.abs(off).argmin()]
                if abs((fu[s] - fu[i]) - (fu[f] - fu[i])) > gap_tol:
                    continue
            else:
                s = sm[np.abs(fu[sm] - fu[i]).argmin()]
            anchors.append(dict(patient=pid, view=view[i], img_a=img[i], lab_a=int(lab[i]),
                                img_f=img[f], lab_f=int(lab[f]), img_s=img[s], lab_s=int(lab[s]),
                                gap_f=int(abs(fu[f] - fu[i])), gap_s=int(abs(fu[s] - fu[i])),
                                sgap_f=int(fu[f] - fu[i]), sgap_s=int(fu[s] - fu[i])))
        if anchors:
            idx = rng.permutation(len(anchors))[:max_anchors_per_patient]
            rows += [anchors[i] for i in idx]
    if same_view and rows:   # a patient can appear in two view groups: keep the per-patient cap
        t = pd.DataFrame(rows).sample(frac=1, random_state=seed)
        rows = t.groupby("patient").head(max_anchors_per_patient).to_dict("records")
    t = pd.DataFrame(rows)
    if len(t):
        t["direction"] = np.where(t["lab_a"] == 0, "absent->present", "present->absent")
    return t


def flip_counts(df, findings=NIH_FINDINGS, same_view=True):
    """Feasibility table: per finding, patients with a within-patient flip (each direction),
    and anchors with BOTH a flip and a same-label partner."""
    out = []
    key = ["Patient ID", "View Position"] if same_view else ["Patient ID"]
    for f in findings:
        d = add_label(df, f).sort_values(key + ["Follow-up #"])
        diff = d.groupby(key)["label"].diff()
        up = d.loc[diff == 1, "Patient ID"].nunique()      # consecutive studies, same view
        down = d.loc[diff == -1, "Patient ID"].nunique()
        trip = build_anchor_triples(df, f, same_view=same_view, max_anchors_per_patient=1)
        out.append(dict(finding=f, patients_absent_to_present=up, patients_present_to_absent=down,
                        patients_with_flip_and_same_partner=len(trip),
                        of_which_anchor_absent=int((trip["lab_a"] == 0).sum()) if len(trip) else 0,
                        of_which_anchor_present=int((trip["lab_a"] == 1).sum()) if len(trip) else 0))
    return pd.DataFrame(out)


def sample_balanced(trip, n, seed=0):
    """Balance the two flip directions so a yes/no bias cannot masquerade as reliance."""
    parts = [g.sample(min(len(g), n // 2), random_state=seed) for _, g in trip.groupby("direction")]
    return pd.concat(parts).reset_index(drop=True)


def _logit(p, eps=1e-4):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))


def item_quantities(res, scale="prob"):
    """res needs p_a, p_f, p_s (P(yes) under anchor, flip partner, same-label partner).
    scale='prob' uses probability differences; scale='logit' uses log-odds differences
    (less compressed near 0/1, makes models with different calibration more comparable)."""
    r = res.copy()
    if scale == "logit":
        for c in ["p_a", "p_f", "p_s", "p_s1"]:
            if c in r:
                r[c] = _logit(r[c].to_numpy())
    r["d_change"] = (r["p_f"] - r["p_a"]).abs()          # sensitivity (should be large)
    ref = r["p_s1"] if "p_s1" in r else r["p_a"]          # pair control: |p(C2) - p(C1)|
    r["d_preserve"] = (r["p_s"] - ref).abs()             # nuisance sensitivity (should be small)
    r["R"] = r["d_change"] - r["d_preserve"]             # evidence-specific reliance
    dy = r["lab_f"] - r["lab_a"]                          # +1 absent->present, -1 present->absent
    r["signed_dir"] = (r["p_f"] - r["p_a"]) * dy          # >0 = moved the right way
    # Directional, time-matched estimand (primary): change toward the new label on the flip pair,
    # minus the same signed change on a same-label pair of the same patient at the same time offset
    r["D_flip"] = r["signed_dir"]
    r["D_ctrl"] = (r["p_s"] - ref) * dy
    r["R_dir"] = r["D_flip"] - r["D_ctrl"]
    # Sign version: only the DIRECTION of movement counts, so any monotone recalibration of the
    # model's scores (saturation, temperature) leaves it unchanged. Ties count 0.
    r["S_flip"] = np.sign(r["p_f"] - r["p_a"]) * dy
    r["S_ctrl"] = np.sign(r["p_s"] - ref) * dy
    r["R_sign"] = r["S_flip"] - r["S_ctrl"]
    return r


DEFAULT_COLS = ("D_flip", "D_ctrl", "R_dir", "S_flip", "S_ctrl", "R_sign", "d_change", "d_preserve", "R")


def cluster_bootstrap(r, cols=DEFAULT_COLS, B=2000, seed=0, by="patient"):
    """Point estimates (mean of patient means) and percentile CIs resampling PATIENTS."""
    rng = np.random.default_rng(seed)
    cols = [c for c in cols if c in r]
    pm = r.groupby(by)[cols].mean()
    est = pm.mean()
    boots = np.empty((B, len(cols)))
    n = len(pm)
    arr = pm.to_numpy()
    for b in range(B):
        boots[b] = arr[rng.integers(0, n, n)].mean(axis=0)
    lo, hi = np.percentile(boots, [2.5, 97.5], axis=0)
    p_le0 = (boots <= 0).mean(axis=0)               # one-sided bootstrap p for H0: quantity <= 0
    return pd.DataFrame({"estimate": est.values, "ci_low": lo, "ci_high": hi, "p_one_sided": p_le0,
                         "n_patients": n}, index=cols)


def summarize(r, B=2000, seed=0):
    from scipy.stats import binomtest
    overall = cluster_bootstrap(r, B=B, seed=seed)
    by_dir = {k: cluster_bootstrap(g, B=B, seed=seed) for k, g in r.groupby("direction")}
    pm = r.groupby("patient")["R_dir" if "R_dir" in r else "R"].mean()
    nz = pm[pm != 0]
    sign_p = binomtest(int((nz > 0).sum()), len(nz), 0.5, alternative="greater").pvalue if len(nz) else np.nan
    acc = ((r["p_a"] > 0.5).astype(int) == r["lab_a"]).mean()
    return overall, by_dir, sign_p, acc


def paired_difference(r1, r2, col="R", B=4000, seed=0):
    """Difference in a per-item quantity between two models scored on the SAME anchors,
    with a patient-level paired bootstrap. Returns estimate, CI and two-sided bootstrap p."""
    rng = np.random.default_rng(seed)
    a = r1.groupby("patient")[col].mean(); b = r2.groupby("patient")[col].mean()
    a, b = a.align(b, join="inner")
    d = (a - b).to_numpy(); n = len(d)
    boots = np.array([d[rng.integers(0, n, n)].mean() for _ in range(B)])
    p = 2 * min((boots <= 0).mean(), (boots >= 0).mean())
    return dict(estimate=d.mean(), ci_low=np.percentile(boots, 2.5), ci_high=np.percentile(boots, 97.5),
                p_boot=min(p, 1.0), n_patients=n)


def gap_balance(t):
    """Check that the control is time-matched to the flip partner."""
    return pd.Series({"mean_gap_flip": t["gap_f"].mean(), "mean_gap_same": t["gap_s"].mean(),
                      "share_same_time_direction": (np.sign(t["sgap_f"]) == np.sign(t["sgap_s"])).mean()
                      if "sgap_f" in t else np.nan})


Writing pilot_lib.py


In [3]:
import os, glob, json, time, gc, shutil, traceback
import numpy as np, pandas as pd
from pilot_lib import *
T0 = time.time()

FINDINGS = ["Effusion", "Atelectasis", "Pneumothorax", "Cardiomegaly"]   # all trained outputs of the CheXpert DenseNet
TERMS = {"Effusion": "a pleural effusion", "Atelectasis": "atelectasis",
         "Pneumothorax": "a pneumothorax", "Cardiomegaly": "cardiomegaly (an enlarged heart)"}
PROMPTS = {  # id: (template, negated?)  -- for negated prompts P(present) = 1 - P(yes)
    "P0": ("Is there {t} in this chest X-ray? Answer yes or no.", False),
    "P1": ("You are an expert radiologist. Look carefully at this chest radiograph. Does it show {t}? "
           "Answer with a single word: yes or no.", False),
    "P2": ("Is this chest X-ray free of {t}? Answer yes or no.", True),
}
SWEEP_FINDINGS = ["Effusion"]   # findings that get all prompts; the others get P0 only
N_ANCHORS = 1500     # DenseNet
N_VLM = 400          # VLM anchors per finding
N_RANDOM = 1000      # random AUC check per finding (DenseNet)
MASS_MIN = 0.9       # sensitivity analysis: drop anchors where yes/no mass < this on any of the 4 images
TIME_BUDGET_H = 11.0
SEED = 0
OUT = "/kaggle/working"
MAX_PIXELS = 512 * 28 * 28
VLMS = [("Qwen/Qwen2.5-VL-7B-Instruct", "float16"), ("google/medgemma-4b-it", "float32")]
COLS = ["img_a", "img_f", "img_s1", "img_s"]

## 1. NIH metadata and images

In [4]:
def find_file(name, top="/kaggle/input"):
    for r, dirs, files in os.walk(top, followlinks=True):
        if name in files:
            return os.path.join(r, name)
        dirs[:] = [d for d in dirs if not d.startswith("images")]
    return None

print("Attached inputs:", os.listdir("/kaggle/input"))
meta = find_file("Data_Entry_2017.csv") or find_file("Data_Entry_2017_v2020.csv")
assert meta, "Attach the NIH Chest X-rays dataset (Add Input -> Datasets -> nih-chest-xrays/data)"
df = pd.read_csv(meta); df.columns = [c.strip() for c in df.columns]
root = os.path.dirname(meta)
img_paths = {os.path.basename(p): p for p in glob.glob(f"{root}/**/*.png", recursive=True)}
print(df.shape, "|", len(img_paths), "images")
assert img_paths, "Images not found - the dataset must be attached, not only the CSV"

Attached inputs: ['datasets']
(112120, 12) | 112120 images


## 2. Re-use cached scores from earlier runs (notebook 03 output or this notebook's own output)

In [5]:
def cache_path(model_tag, finding=None, pid=None):
    return f"{OUT}/scores04_xrv.json" if finding is None else f"{OUT}/scores04_{finding}_{model_tag}_{pid}.json"

def load(p):
    return json.load(open(p)) if os.path.exists(p) else {}

found = []
for r, dirs, files in os.walk("/kaggle/input", followlinks=True):
    dirs[:] = [d for d in dirs if not d.startswith("images")]
    for f in files:
        if f.startswith("scores04_") and f.endswith(".json") and not os.path.exists(f"{OUT}/{f}"):
            shutil.copy(os.path.join(r, f), f"{OUT}/{f}"); found.append(f)
        # notebook 03: Effusion, prompt P0 (identical wording) -> merge into the P0 caches
        if f.startswith("scores03_Effusion_") and f.endswith(".json") and "xrv" not in f:
            tag = f[len("scores03_Effusion_"):-5]
            dst = cache_path(tag, "Effusion", "P0"); c = load(dst); c.update(json.load(open(os.path.join(r, f))))
            json.dump(c, open(dst, "w")); found.append(f + " -> " + os.path.basename(dst))
print("re-used caches:", found or "none")

re-used caches: ['scores03_Effusion_Qwen2.5-VL-7B-Instruct.json -> scores04_Effusion_Qwen2.5-VL-7B-Instruct_P0.json', 'scores03_Effusion_medgemma-4b-it.json -> scores04_Effusion_medgemma-4b-it_P0.json', 'scores04_Atelectasis_Qwen2.5-VL-7B-Instruct_P0.json', 'scores04_Effusion_Qwen2.5-VL-7B-Instruct_P1.json', 'scores04_Cardiomegaly_Qwen2.5-VL-7B-Instruct_P0.json', 'scores04_Effusion_Qwen2.5-VL-7B-Instruct_P2.json', 'scores04_xrv.json', 'scores04_Pneumothorax_Qwen2.5-VL-7B-Instruct_P0.json']


## 3. Anchors per finding (time-matched pair controls) and random AUC sets

In [6]:
samps, rands = {}, {}
for F in FINDINGS:
    trip = build_anchor_triples(df, F, same_view=True, max_anchors_per_patient=1, seed=SEED, control="pair")
    if len(trip) == 0 or trip["direction"].nunique() < 2:
        print(F, "- no usable anchors, skipped"); continue
    trip = trip[trip[COLS].isin(list(img_paths)).all(axis=1)]
    samp = sample_balanced(trip, N_ANCHORS, seed=SEED).reset_index(drop=True)
    # same construction as notebook 03 (800-anchor VLM set), then a balanced 400 subset of it
    v800 = sample_balanced(samp.assign(_i=samp.index), 800, seed=SEED + 1)
    samp["vlm"] = samp.index.isin(sample_balanced(v800, N_VLM, seed=SEED + 2)["_i"])
    samps[F] = samp
    lab_all = add_label(df, F)
    lab_all = lab_all[~lab_all["Patient ID"].isin(samp["patient"]) & lab_all["Image Index"].isin(list(img_paths))]
    one = lab_all.sample(frac=1, random_state=SEED).drop_duplicates("Patient ID")
    rands[F] = pd.concat([one[one["label"] == 1].head(N_RANDOM // 2), one[one["label"] == 0].head(N_RANDOM // 2)])
    samp.to_csv(f"{OUT}/triples04_{F}.csv", index=False)
    print(f"{F:13s} available {len(trip):5d} {trip['direction'].value_counts().to_dict()} | DenseNet anchors {len(samp)}"
          f" | VLM anchors {int(samp['vlm'].sum())} | gap balance {gap_balance(samp).round(2).to_dict()}")
FINDINGS = [F for F in FINDINGS if F in samps]; SWEEP_FINDINGS = [F for F in SWEEP_FINDINGS if F in samps]
print("NOTE: anchors are capped by the rarer direction (present->absent); a small n for a finding means wide CIs, not a bug.")

Effusion      available  2713 {'absent->present': 2018, 'present->absent': 695} | DenseNet anchors 1445 | VLM anchors 400 | gap balance {'mean_gap_flip': 2.92, 'mean_gap_same': 2.92, 'share_same_time_direction': 1.0}
Atelectasis   available  2849 {'absent->present': 2258, 'present->absent': 591} | DenseNet anchors 1341 | VLM anchors 400 | gap balance {'mean_gap_flip': 2.87, 'mean_gap_same': 2.87, 'share_same_time_direction': 1.0}
Pneumothorax  available  1167 {'absent->present': 883, 'present->absent': 284} | DenseNet anchors 1034 | VLM anchors 400 | gap balance {'mean_gap_flip': 3.92, 'mean_gap_same': 3.92, 'share_same_time_direction': 1.0}
Cardiomegaly  available   740 {'absent->present': 642, 'present->absent': 98} | DenseNet anchors 740 | VLM anchors 298 | gap balance {'mean_gap_flip': 5.59, 'mean_gap_same': 5.59, 'share_same_time_direction': 1.0}
NOTE: anchors are capped by the rarer direction (present->absent); a small n for a finding means wide CIs, not a bug.


## 4. Scorers

In [7]:
import torch
from PIL import Image
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("No HF_TOKEN secret - MedGemma will be skipped")
print("GPUs:", torch.cuda.device_count())

def make_xrv():
    import torchxrayvision as xrv, torchvision, skimage.io
    m = xrv.models.DenseNet(weights="densenet121-res224-chex").eval()
    dev = "cuda" if torch.cuda.is_available() else "cpu"; m = m.to(dev)
    idx = {F: m.targets.index(F) for F in FINDINGS}     # fails loudly if a finding is not a trained output
    tfm = torchvision.transforms.Compose([xrv.datasets.XRayCenterCrop(), xrv.datasets.XRayResizer(224)])
    @torch.no_grad()
    def score(path):
        img = skimage.io.imread(path)
        if img.ndim == 3:
            img = img[..., :3].mean(2)
        img = xrv.utils.normalize(img, 255 if img.max() <= 255 else 65535)
        out = m(torch.from_numpy(tfm(img[None])).float()[None].to(dev))[0]
        return {F: float(out[i]) for F, i in idx.items()}
    return score, m

def make_vlm(name, dtype):
    from transformers import AutoProcessor
    try:
        from transformers import AutoModelForImageTextToText as Auto
    except ImportError:
        from transformers import AutoModelForVision2Seq as Auto
    dt = getattr(torch, dtype)
    extra = {"attn_implementation": "eager"} if "gemma" in name.lower() else {}
    m = Auto.from_pretrained(name, torch_dtype=dt, device_map="auto", token=token, **extra).eval()
    kw = {"max_pixels": MAX_PIXELS} if "Qwen" in name else {}
    proc = AutoProcessor.from_pretrained(name, token=token, **kw)
    tok = proc.tokenizer
    ids = lambda ws: sorted({tok.encode(w, add_special_tokens=False)[0] for w in ws})
    YES, NO = ids(["Yes", "yes", " Yes", " yes"]), ids(["No", "no", " No", " no"])
    dev = next(m.parameters()).device

    def inputs(img, question):
        msgs = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": question}]}]
        try:
            inp = proc.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True,
                                           return_dict=True, return_tensors="pt")
        except Exception:
            msgs[0]["content"][0] = {"type": "image"}
            text = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            inp = proc(text=[text], images=[img], return_tensors="pt")
        out = {}
        for k, v in inp.items():
            if torch.is_tensor(v):
                v = v.to(dev)
                if v.is_floating_point():
                    v = v.to(dt)
            out[k] = v
        return out

    @torch.no_grad()
    def score(path, question):
        img = Image.open(path).convert("RGB")
        lp = torch.log_softmax(m(**inputs(img, question)).logits[0, -1].float(), -1)
        ly, ln = torch.logsumexp(lp[YES], 0), torch.logsumexp(lp[NO], 0)
        return torch.sigmoid(ly - ln).item(), (ly.exp() + ln.exp()).item()   # raw P(yes), yes/no mass
    return score, m

def out_of_time():
    return (time.time() - T0) / 3600 > TIME_BUDGET_H

GPUs: 2


## 5. Score images (cached and resumable; full error printed on failure)

In [8]:
def anchor_images(F, vlm_only):
    s = samps[F][samps[F]["vlm"]] if vlm_only else samps[F]
    return list(pd.unique(s[COLS].to_numpy().ravel()))

# DenseNet: every anchor image and random image of every finding (all outputs in one pass)
xc = load(cache_path("xrv"))
todo = sorted({im for F in FINDINGS for im in anchor_images(F, False) + list(rands[F]["Image Index"])} - set(xc))
print("DenseNet:", len(todo), "images to score")
if todo:
    score, model = make_xrv(); t = time.time()
    for i, im in enumerate(todo):
        xc[im] = score(img_paths[im])
        if i % 1000 == 0: print(f"  {i}/{len(todo)} {time.time() - t:.0f}s")
    json.dump(xc, open(cache_path("xrv"), "w")); del model; gc.collect(); torch.cuda.empty_cache()

# VLMs: P0 on every finding first, then the extra prompts on the sweep findings
jobs = [(F, "P0") for F in FINDINGS] + [(F, p) for F in SWEEP_FINDINGS for p in PROMPTS if p != "P0"]
timing = []
for name, dtype in VLMS:
    tag = name.split("/")[-1]
    if "medgemma" in name and token is None:
        print("skip", name); continue
    pending = [(F, p) for F, p in jobs if set(anchor_images(F, True)) - set(load(cache_path(tag, F, p)))]
    if not pending or out_of_time():
        print(tag, "nothing to do" if not pending else "out of time"); continue
    model = None
    try:
        score, model = make_vlm(name, dtype)
        for F, p in pending:
            q = PROMPTS[p][0].format(t=TERMS[F]); cp = cache_path(tag, F, p); c = load(cp)
            todo = [im for im in anchor_images(F, True) if im not in c]
            print(f"{tag} | {F} | {p} | {len(todo)} images | {q}")
            t = time.time()
            for i, im in enumerate(todo):
                if out_of_time():
                    print("  time budget reached - stopping, results so far are kept"); break
                c[im] = score(img_paths[im], q)
                if i % 200 == 0:
                    json.dump(c, open(cp, "w")); print(f"  {i}/{len(todo)} {time.time() - t:.0f}s")
            json.dump(c, open(cp, "w"))
            timing.append(dict(model=tag, finding=F, prompt=p, n=len(todo), sec_per_img=(time.time() - t) / max(len(todo), 1)))
    except Exception:
        print("FAILED", name); traceback.print_exc()
    finally:
        del model; gc.collect(); torch.cuda.empty_cache()
pd.DataFrame(timing).to_csv(f"{OUT}/timing04.csv", index=False); print(pd.DataFrame(timing))

DenseNet: 0 images to score
Qwen2.5-VL-7B-Instruct nothing to do


config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

medgemma-4b-it | Atelectasis | P0 | 1402 images | Is there atelectasis in this chest X-ray? Answer yes or no.
  0/1402 4s
  200/1402 748s
  400/1402 1507s
  600/1402 2266s
  800/1402 3026s
  1000/1402 3786s
  1200/1402 4546s
  1400/1402 5308s
medgemma-4b-it | Pneumothorax | P0 | 1435 images | Is there a pneumothorax in this chest X-ray? Answer yes or no.
  0/1435 4s
  200/1435 765s
  400/1435 1526s
  600/1435 2288s
  800/1435 3050s
  1000/1435 3812s
  1200/1435 4574s
  1400/1435 5336s
medgemma-4b-it | Cardiomegaly | P0 | 1081 images | Is there cardiomegaly (an enlarged heart) in this chest X-ray? Answer yes or no.
  0/1081 4s
  200/1081 766s
  400/1081 1528s
  600/1081 2290s
  800/1081 3051s
  1000/1081 3813s
medgemma-4b-it | Effusion | P1 | 1428 images | You are an expert radiologist. Look carefully at this chest radiograph. Does it show a pleural effusion? Answer with a single word: yes or no.
  0/1428 4s
  200/1428 767s
  400/1428 1529s
  600/1428 2291s
  800/1428 3053s
  1000/1428 

## 6. Reliance estimates per finding x model x prompt (probability scale)

In [9]:
from sklearn.metrics import roc_auc_score

def get_scores(tag, F, p):
    """image -> (P(finding present), yes/no mass). Negated prompts are flipped so 'higher = present' always."""
    if tag == "xrv":
        return {k: (v[F], 1.0) for k, v in xc.items() if F in v}
    c = load(cache_path(tag, F, p))
    return {k: ((1 - v[0]) if PROMPTS[p][1] else v[0], v[1]) for k, v in c.items()}

def items(tag, F, p, vlm_only, mass_min=None):
    sc = get_scores(tag, F, p)
    s = samps[F][samps[F]["vlm"]] if vlm_only else samps[F]
    s = s[s[COLS].apply(lambda r: all(k in sc for k in r), axis=1)].copy()
    for c in ["a", "f", "s1", "s"]:
        s[f"p_{c}"] = s[f"img_{c}"].map(lambda k: sc[k][0]); s[f"m_{c}"] = s[f"img_{c}"].map(lambda k: sc[k][1])
    if mass_min is not None:
        s = s[s[[f"m_{c}" for c in ["a", "f", "s1", "s"]]].min(axis=1) >= mass_min]
    return item_quantities(s, scale="prob") if len(s) else None

configs = [("xrv", F, "-", sub) for F in FINDINGS for sub in ["all", "vlm"]]
configs += [(n.split("/")[-1], F, p, "vlm") for n, _ in VLMS for F, p in jobs]
rows, R = [], {}
for tag, F, p, sub in configs:
    for filt in ([None] if tag == "xrv" else [None, MASS_MIN]):
        r = items(tag, F, p, sub == "vlm", filt)
        if r is None or r["patient"].nunique() < 30:
            continue
        R[(tag, F, p, sub, filt)] = r
        ov, by_dir, sign_p, _ = summarize(r, B=2000, seed=SEED)
        img = pd.concat([r[["p_a", "lab_a"]].set_axis(["p", "y"], axis=1), r[["p_f", "lab_f"]].set_axis(["p", "y"], axis=1)])
        row = dict(finding=F, model=tag, prompt=p, anchors=sub, mass_filter=filt or "-", n_patients=r["patient"].nunique(),
                   AUC_flip_imgs=roc_auc_score(img["y"], img["p"]), yes_rate=(img["p"] > 0.5).mean(), sign_test_p=sign_p)
        for q in ["R_dir", "D_flip", "D_ctrl", "R_sign"]:
            row[q] = ov.loc[q, "estimate"]; row[q + "_lo"] = ov.loc[q, "ci_low"]; row[q + "_hi"] = ov.loc[q, "ci_high"]
        row["p(R_dir<=0)"] = ov.loc["R_dir", "p_one_sided"]
        for d, v in by_dir.items():
            row["R_dir " + d] = v.loc["R_dir", "estimate"]
        rows.append(row)
        if filt is None:
            r.to_csv(f"{OUT}/items04_{F}_{tag}_{p}_{sub}.csv", index=False)
summary = pd.DataFrame(rows); summary.to_csv(f"{OUT}/summary04.csv", index=False)
pd.set_option("display.width", 300); pd.set_option("display.max_columns", 40)
display(summary[summary["mass_filter"] == "-"].round(3))

,finding,model,prompt,anchors,mass_filter,n_patients,AUC_flip_imgs,yes_rate,sign_test_p,R_dir,R_dir_lo,R_dir_hi,D_flip,D_flip_lo,D_flip_hi,D_ctrl,D_ctrl_lo,D_ctrl_hi,R_sign,R_sign_lo,R_sign_hi,p(R_dir<=0),R_dir absent->present,R_dir present->absent
0,Effusion,xrv,-,all,-,1445,0.576,0.683,0.000,0.063,0.041,0.082,0.091,0.075,0.105,0.028,0.015,0.042,0.176,0.097,0.249,0.000,0.051,0.075
1,Effusion,xrv,-,vlm,-,400,0.579,0.696,0.003,0.080,0.041,0.119,0.097,0.070,0.126,0.017,-0.011,0.046,0.250,0.120,0.385,0.000,0.051,0.108
2,Atelectasis,xrv,-,all,-,1341,0.550,0.736,0.036,0.047,0.028,0.065,0.061,0.048,0.074,0.014,0.002,0.027,0.101,0.027,0.178,0.000,0.040,0.055
3,Atelectasis,xrv,-,vlm,-,400,0.531,0.746,0.480,0.011,-0.020,0.043,0.039,0.016,0.062,0.027,0.004,0.050,0.108,-0.035,0.250,0.238,-0.009,0.032
4,Pneumothorax,xrv,-,all,-,1034,0.535,0.288,0.101,0.016,-0.000,0.030,0.020,0.009,0.032,0.005,-0.006,0.015,0.048,-0.041,0.137,0.026,0.006,0.042
5,Pneumothorax,xrv,-,vlm,-,400,0.546,0.294,0.125,0.016,-0.009,0.041,0.026,0.008,0.044,0.009,-0.006,0.026,0.080,-0.075,0.230,0.094,-0.021,0.054
6,Cardiomegaly,xrv,-,all,-,740,0.552,0.663,0.291,0.028,-0.001,0.054,0.059,0.037,0.080,0.032,0.012,0.051,0.078,-0.022,0.173,0.028,0.040,-0.053
7,Cardiomegaly,xrv,-,vlm,-,298,0.536,0.685,0.386,0.007,-0.035,0.050,0.040,0.009,0.074,0.033,0.003,0.064,0.081,-0.074,0.242,0.372,0.037,-0.053
8,Effusion,Qwen2.5-VL-7B-Instruct,P0,vlm,-,400,0.515,0.000,0.599,0.001,-0.006,0.008,0.002,-0.003,0.006,0.001,-0.004,0.005,0.045,-0.100,0.190,0.424,0.003,-0.002
10,Atelectasis,Qwen2.5-VL-7B-Instruct,P0,vlm,-,400,0.497,0.025,0.520,-0.009,-0.033,0.014,-0.005,-0.020,0.011,0.004,-0.010,0.018,-0.048,-0.200,0.102,0.800,-0.025,0.007


## 7. Model comparisons on shared anchors (prompt P0, probability scale; R_dir and R_sign; BH over all tests)

In [10]:
comp = []
for F in FINDINGS:
    keys = [k for k in R if k[1] == F and k[3] == "vlm" and k[4] is None and k[2] in ("-", "P0")]
    for i in range(len(keys)):
        for j in range(i + 1, len(keys)):
            for col in ["R_dir", "R_sign"]:
                d = paired_difference(R[keys[i]], R[keys[j]], col=col, seed=SEED)
                comp.append(dict(finding=F, model_1=keys[i][0], model_2=keys[j][0], estimand=col, **d))
comp = pd.DataFrame(comp)
if len(comp):
    p = comp["p_boot"].to_numpy(); o = np.argsort(p); m = len(p)
    adj = np.minimum.accumulate((p[o] * m / np.arange(1, m + 1))[::-1])[::-1]
    comp["p_fdr"] = np.empty(m); comp.loc[o, "p_fdr"] = np.minimum(adj, 1)
    comp.to_csv(f"{OUT}/paired04.csv", index=False); display(comp.round(4))

,finding,model_1,model_2,estimand,estimate,ci_low,ci_high,p_boot,n_patients,p_fdr
0,Effusion,xrv,Qwen2.5-VL-7B-Instruct,R_dir,0.0787,0.0397,0.1184,0.0000,400,0.0000
1,Effusion,xrv,Qwen2.5-VL-7B-Instruct,R_sign,0.2050,0.0300,0.3950,0.0200,400,0.0600
2,Effusion,xrv,medgemma-4b-it,R_dir,-0.0806,-0.1458,-0.0157,0.0145,400,0.0497
3,Effusion,xrv,medgemma-4b-it,R_sign,-0.0125,-0.1600,0.1425,0.8920,400,0.9475
4,Effusion,Qwen2.5-VL-7B-Instruct,medgemma-4b-it,R_dir,-0.1593,-0.2257,-0.0910,0.0000,400,0.0000
5,Effusion,Qwen2.5-VL-7B-Instruct,medgemma-4b-it,R_sign,-0.2175,-0.4125,-0.0275,0.0275,400,0.0733
6,Atelectasis,xrv,Qwen2.5-VL-7B-Instruct,R_dir,0.0207,-0.0157,0.0570,0.2580,400,0.3259
7,Atelectasis,xrv,Qwen2.5-VL-7B-Instruct,R_sign,0.1550,-0.0350,0.3500,0.1190,400,0.2197
8,Atelectasis,xrv,medgemma-4b-it,R_dir,-0.0493,-0.0962,-0.0028,0.0355,400,0.0852
9,Atelectasis,xrv,medgemma-4b-it,R_sign,-0.0925,-0.2750,0.0950,0.3445,400,0.4134


## 8. Prompt sensitivity (sweep findings): does the conclusion depend on wording?

In [11]:
ps = []
for F in SWEEP_FINDINGS:
    for n, _ in VLMS:
        tag = n.split("/")[-1]
        base = R.get((tag, F, "P0", "vlm", None))
        for p in PROMPTS:
            if p == "P0" or base is None or (tag, F, p, "vlm", None) not in R:
                continue
            for col in ["R_dir", "R_sign"]:
                d = paired_difference(R[(tag, F, p, "vlm", None)], base, col=col, seed=SEED)
                ps.append(dict(finding=F, model=tag, prompt=p, vs="P0", estimand=col, **d))
ps = pd.DataFrame(ps); ps.to_csv(f"{OUT}/prompt04.csv", index=False)
display(ps.round(4))
display(summary[(summary["finding"].isin(SWEEP_FINDINGS)) & (summary["mass_filter"] == "-")]
        [["finding", "model", "prompt", "n_patients", "AUC_flip_imgs", "yes_rate", "R_dir", "R_dir_lo", "R_dir_hi", "R_sign"]].round(3))

,finding,model,prompt,vs,estimand,estimate,ci_low,ci_high,p_boot,n_patients
0,Effusion,Qwen2.5-VL-7B-Instruct,P1,P0,R_dir,0.0001,-0.0094,0.0097,0.9780,400
1,Effusion,Qwen2.5-VL-7B-Instruct,P1,P0,R_sign,-0.0100,-0.0850,0.0650,0.8500,400
2,Effusion,Qwen2.5-VL-7B-Instruct,P2,P0,R_dir,0.0103,-0.0073,0.0273,0.2565,400
3,Effusion,Qwen2.5-VL-7B-Instruct,P2,P0,R_sign,0.0650,-0.0300,0.1550,0.2100,400
4,Effusion,medgemma-4b-it,P1,P0,R_dir,0.0215,-0.0113,0.0545,0.1835,400
5,Effusion,medgemma-4b-it,P1,P0,R_sign,0.0100,-0.0775,0.1000,0.8520,400
6,Effusion,medgemma-4b-it,P2,P0,R_dir,-0.0703,-0.1298,-0.0131,0.0180,400
7,Effusion,medgemma-4b-it,P2,P0,R_sign,-0.0025,-0.1375,0.1350,0.9915,400


,finding,model,prompt,n_patients,AUC_flip_imgs,yes_rate,R_dir,R_dir_lo,R_dir_hi,R_sign
0,Effusion,xrv,-,1445,0.576,0.683,0.063,0.041,0.082,0.176
1,Effusion,xrv,-,400,0.579,0.696,0.080,0.041,0.119,0.250
8,Effusion,Qwen2.5-VL-7B-Instruct,P0,400,0.515,0.000,0.001,-0.006,0.008,0.045
16,Effusion,Qwen2.5-VL-7B-Instruct,P1,400,0.518,0.006,0.001,-0.015,0.017,0.035
18,Effusion,Qwen2.5-VL-7B-Instruct,P2,400,0.527,0.798,0.011,-0.010,0.033,0.110
20,Effusion,medgemma-4b-it,P0,400,0.645,0.586,0.160,0.094,0.225,0.262
28,Effusion,medgemma-4b-it,P1,400,0.653,0.552,0.182,0.120,0.248,0.272
30,Effusion,medgemma-4b-it,P2,400,0.611,0.764,0.090,0.039,0.142,0.260


## 9. DenseNet random-image AUC per finding (label-noise check, as in notebook 03)

In [12]:
diag = []
for F in FINDINGS:
    rr = rands[F]; p = rr["Image Index"].map(lambda k: xc[k][F])
    flip = summary[(summary.finding == F) & (summary.model == "xrv") & (summary.anchors == "all")]
    diag.append(dict(finding=F, AUC_random=roc_auc_score(rr["label"], p),
                     AUC_flip_imgs=flip["AUC_flip_imgs"].iloc[0] if len(flip) else np.nan, n_random=len(rr)))
diag = pd.DataFrame(diag); diag.to_csv(f"{OUT}/diagnostic04.csv", index=False); display(diag.round(3))
print(f"total runtime {(time.time() - T0) / 3600:.2f} h")

,finding,AUC_random,AUC_flip_imgs,n_random
0,Effusion,0.768,0.576,1000
1,Atelectasis,0.653,0.550,1000
2,Pneumothorax,0.575,0.535,691
3,Cardiomegaly,0.619,0.552,1000


total runtime 7.23 h


## 10. Reading the results
1. **Generality (section 6/7):** does MedGemma > DenseNet > Qwen-7B hold on every finding? A reversal on one finding is a
   real result, not a failure - report it.
2. **Pneumothorax:** a low R_dir with a decent AUC would be the first sign of the accuracy/reliance *dissociation*
   the paper needs (e.g. reacting to chest drains rather than to the pneumothorax itself).
3. **Prompt sensitivity (section 8):** if Qwen-7B stays ~0 under all three wordings, its null is not a prompt failure.
   With the negated prompt P2, a model with a strong yes-bias shows yes_rate near 1 under P0 and near 0 under P2.
   If R_dir changes a lot between wordings, the estimand must be reported per prompt, or averaged over a prompt set.
4. **R_sign vs R_dir:** the same ordering under both means calibration is not driving the comparison.
5. **Mass filter rows:** results should barely move; if they do, the low-mass items were driving them.

**Send back:** zip of `/kaggle/working` (summary04, paired04, prompt04, diagnostic04, timing04, items04, triples04, scores04)
and the full text of any `FAILED` traceback.